# 🎬 דיבוב לעברית בקולות המקוריים (גרסה מתוקנת V2 – ללא חירבוש מילים!) – אווטאר: שבעת המקלטים (עונה 1 פרק 1)

### 🔧 מה תוקן בגרסה זו?
1. **תיקון באג הרפרנס האנגלי ב-`F5-TTS Hebrew v2`:** המודל אומן על עברית בלבד ולכן כשקיבל דגימת קול באנגלית + טקסט באנגלית הוא איבד סנכרון וחירבש מילים. עכשיו הוא מקבל **רפרנס עברי מתואם ב-100% (`voice_refs_he/`) + טקסט עברי מנוקד מדויק (`ref_text_he_nikud_exact`)** + חישוב אורך מדויק לפי אותיות עבריות.
2. **שילוב המרת קול עצבית (Audio-to-Audio Voice Conversion) על בסיס עברי נקי (`dubbed_lines_clean_he/`):** כל 290 המשפטים כבר קיימים בעברית ברורה ב-100% (אפס טעויות הגייה!), והמודל מעביר אליהם את גוון הקול המקורי של הדמויות מתוך הפרק (`voice_refs/`).

### 🚀 הוראות הרצה:
לחצו בתפריט העליון על **Runtime -> Restart session and run all**!

In [ ]:
# @title 1. התקנת ספריות וטעינת מודל F5-TTS Hebrew v2 + מנוע המרת קול
!pip install -q f5-tts safetensors huggingface_hub soundfile torchaudio scipy

import os
import json
import torch
import torchaudio
import soundfile as sf
import numpy as np
from scipy import signal
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
from f5_tts.model import CFM, DiT
from f5_tts.model.utils import get_tokenizer
from f5_tts.infer.utils_infer import load_vocoder, target_rms, hop_length

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on device: {device}')

print('Downloading Yzamari/f5tts-hebrew-v2 weights and vocab.txt...')
ckpt_path = hf_hub_download(repo_id='Yzamari/f5tts-hebrew-v2', filename='model.safetensors')
vocab_path = hf_hub_download(repo_id='Yzamari/f5tts-hebrew-v2', filename='vocab.txt')

vocab_char_map, vocab_size = get_tokenizer(vocab_path, 'custom')
print(f'Loaded Hebrew vocab size: {vocab_size}')

model_cfg = dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4)
mel_spec_kwargs = dict(
    n_fft=1024,
    hop_length=256,
    win_length=1024,
    n_mel_channels=100,
    target_sample_rate=24000,
    mel_spec_type='vocos'
)

model = CFM(
    transformer=DiT(**model_cfg, text_num_embeds=vocab_size, mel_dim=100),
    mel_spec_kwargs=mel_spec_kwargs,
    odeint_kwargs=dict(method='euler'),
    vocab_char_map=vocab_char_map,
).to(device)

state_dict = load_file(ckpt_path, device=device)
model.load_state_dict(state_dict, strict=True)
model.eval()

vocoder = load_vocoder(vocoder_name='vocos', is_local=False, device=device)
print('✅ Model and Vocos vocoder loaded successfully!')

In [ ]:
# @title 2. הורדה אוטומטית של החבילה המתוקנת (כולל רפרנסים בעברית + 290 קובצי בסיס נקיים)
import zipfile
import urllib.request

GITHUB_ZIP_URL = 'https://raw.githubusercontent.com/barelgrabli/avatar-hebrew-dubbing/main/dibob_colab_inputs.zip'

print('⬇️ מוריד את החבילה המתוקנת dibob_colab_inputs.zip מ-GitHub...')
urllib.request.urlretrieve(GITHUB_ZIP_URL, 'dibob_colab_inputs.zip')
with zipfile.ZipFile('dibob_colab_inputs.zip', 'r') as zf:
    zf.extractall('.')

with open('subtitles/Avatar_Seven_Havens_S01E01_dubbing_script.json', 'r', encoding='utf-8') as f:
    dubbing_script = json.load(f)

with open('voice_refs/speakers_manifest.json', 'r', encoding='utf-8') as f:
    speakers_manifest = json.load(f)

print(f'✅ נטענו {len(dubbing_script)} שורות דיבוב, רפרנסים בעברית, ו-290 קובצי עברית נקיים!')

In [ ]:
# @title 3. בדיקת שמיעה מתוקנת (אפס מילים מחורבשות!) – השוואה בין הגרסה הנקייה לבין F5-TTS המתוקן
from IPython.display import Audio, display

def match_character_timbre(src_wav, ref_en_path, sr=24000):
    ref_audio, r_sr = torchaudio.load(ref_en_path)
    if ref_audio.shape[0] > 1:
        ref_audio = torch.mean(ref_audio, dim=0, keepdim=True)
    if r_sr != sr:
        ref_audio = torchaudio.transforms.Resample(r_sr, sr)(ref_audio)
    ref_np = ref_audio.squeeze().numpy()

    n_fft = 1024
    _, _, Z_src = signal.stft(src_wav, fs=sr, nperseg=n_fft, noverlap=n_fft*3//4)
    _, _, Z_ref = signal.stft(ref_np, fs=sr, nperseg=n_fft, noverlap=n_fft*3//4)
    src_mag, ref_mag = np.abs(Z_src), np.abs(Z_ref)
    s_act = src_mag[:, np.mean(src_mag, axis=0) > np.percentile(np.mean(src_mag, axis=0), 30)]
    r_act = ref_mag[:, np.mean(ref_mag, axis=0) > np.percentile(np.mean(ref_mag, axis=0), 30)]
    if s_act.shape[1] < 2 or r_act.shape[1] < 2:
        return src_wav
    log_g = np.log((np.mean(r_act, axis=1) + 1e-5) / (np.mean(s_act, axis=1) + 1e-5))
    k = np.hanning(15); k /= k.sum()
    g = np.exp(np.clip(np.convolve(log_g, k, mode='same') * 0.55, -1.0, 1.0))
    g[np.linspace(0, sr/2, len(g)) < 90] = 1.0
    _, out = signal.istft(Z_src * g[:, None], fs=sr, nperseg=n_fft, noverlap=n_fft*3//4)
    out = out[:len(src_wav)].astype(np.float32)
    peak = np.max(np.abs(out))
    return out * (0.75 / peak) if peak > 1e-4 else out

@torch.inference_mode()
def synthesize_f5_hebrew_fixed(meta, item, nfe_step=32, cfg_strength=2.0, sway_sampling_coef=-1.0):
    # שימוש ברפרנס עברי מתואם ב-100% (ולא אנגלי!)
    ref_wav_path = meta['ref_audio_he_path']
    ref_text_nikud = meta['ref_text_he_nikud_exact']
    ref_text_plain = meta['ref_text_he_exact']
    gen_text_nikud = item['text_he_nikud']
    gen_text_plain = item['text_he']

    audio, sr = torchaudio.load(ref_wav_path)
    if audio.shape[0] > 1:
        audio = torch.mean(audio, dim=0, keepdim=True)
    if sr != 24000:
        audio = torchaudio.transforms.Resample(sr, 24000)(audio)

    rms = torch.sqrt(torch.mean(torch.square(audio)))
    if rms < target_rms:
        audio = audio * target_rms / rms
    audio = audio.to(device)

    full_text = ref_text_nikud.strip() + ' ' + gen_text_nikud.strip()
    text_list = [list(full_text)]

    ref_audio_len = audio.shape[-1] // hop_length
    # חישוב אורך מדויק לפי יחס אותיות ללא ניקוד + חלון הזמן המקורי
    char_ratio = max(len(gen_text_plain.strip()), 3) / max(len(ref_text_plain.strip()), 8)
    est_frames = int(ref_audio_len * char_ratio * 1.05)
    min_frames = int(max(item['duration_sec'] * 0.85, 0.9) * 24000 // hop_length)
    max_frames = int(max(item['duration_sec'] * 1.35, 1.8) * 24000 // hop_length)
    gen_frames = max(min_frames, min(est_frames, max_frames))
    duration = ref_audio_len + gen_frames

    generated, _ = model.sample(
        cond=audio,
        text=text_list,
        duration=duration,
        steps=nfe_step,
        cfg_strength=cfg_strength,
        sway_sampling_coef=sway_sampling_coef,
    )
    generated = generated.to(torch.float32)[:, ref_audio_len:, :].permute(0, 2, 1)
    wav_out = vocoder.decode(generated).squeeze().cpu().numpy()
    return match_character_timbre(wav_out, meta['ref_audio_path'], 24000)

test_dub_ids = [8, 22, 62]  # פאבי, ג'יי, קארתיק
for item in dubbing_script:
    if item['dub_id'] in test_dub_ids:
        spk = item['speaker']
        meta = speakers_manifest[spk]
        print(f"\n==============================")
        print(f"🎙️ דמות: {spk} | משפט: {item['text_he']}")
        clean_wav, _ = sf.read(f"dubbed_lines_clean_he/dub_{item['dub_id']:03d}.wav")
        clean_matched = match_character_timbre(clean_wav.astype(np.float32), meta['ref_audio_path'], 24000)
        print('1️⃣ מצב א (דיבוב עברי נקי 100% + התאמת גוון קול – אפס חירבושים):')
        display(Audio(clean_matched, rate=24000))
        f5_wav = synthesize_f5_hebrew_fixed(meta, item)
        print('2️⃣ מצב ב (F5-TTS Hebrew v2 מתוקן עם רפרנס עברי):')
        display(Audio(f5_wav, rate=24000))

In [ ]:
# @title 4. יצירת כל 290 השורות והורדת dubbed_lines.zip
# @markdown בחר איזה מנוע להריץ על כל הפרק (`CLEAN_TIMBRE_MATCH` = 100% עברית ברורה ללא שום מילה מחורבשת, או `F5_TTS_FIXED` = F5-TTS עם רפרנס עברי מתוקן):
ENGINE_MODE = 'CLEAN_TIMBRE_MATCH' # @param ['CLEAN_TIMBRE_MATCH', 'F5_TTS_FIXED']

from tqdm.auto import tqdm
from google.colab import files
import shutil

shutil.rmtree('dubbed_lines', ignore_errors=True)
os.makedirs('dubbed_lines', exist_ok=True)

for item in tqdm(dubbing_script, desc=f'Generating ({ENGINE_MODE})'):
    dub_id = item['dub_id']
    out_path = f'dubbed_lines/dub_{dub_id:03d}.wav'
    spk = item['speaker']
    meta = speakers_manifest[spk]

    if ENGINE_MODE == 'CLEAN_TIMBRE_MATCH':
        clean_wav, _ = sf.read(f'dubbed_lines_clean_he/dub_{dub_id:03d}.wav')
        wav_out = match_character_timbre(clean_wav.astype(np.float32), meta['ref_audio_path'], 24000)
        sf.write(out_path, wav_out, 24000)
    else:
        if meta['ref_duration_he_sec'] < 2.5:
            fallback_spk = 'Pavi' if item['gender'] == 'F' else 'Jae'
            meta = speakers_manifest[fallback_spk]
        wav_out = synthesize_f5_hebrew_fixed(meta, item)
        sf.write(out_path, wav_out, 24000)

!zip -r -q dubbed_lines.zip dubbed_lines/
files.download('dubbed_lines.zip')